# TWDB GAM CKAN Registration

This notebook registers or refreshes CKAN datasets for every TWDB GAM package.

## Workflow — two paths available

### Path A  (NEW — Capability A/B/D pipeline, recommended)
Runs on Lonestar6 or any host with the Corral GAM filesystem mounted at `GAM_ROOT_DIR`.

1. **Configuration** — set `GAM_ROOT_DIR`, LLM credentials, `MAX_ROUNDS`, `LLM_CALL_DELAY_SECONDS`.
2. **Discovery** (`Section 3.5`) — calls `discovery.discover_gam_models_from_local` to produce a fresh manifest from a recursive `.nam` walk.  Bbox derivation chain: DIS via flopy → aquifer-boundary fallback → null.  Human review checkpoint follows.
3. **CKAN Auth** — unchanged.
4. **Load Manifest** — load the generated (or hand-edited) manifest.
5. **SUBSIDE Pipeline** (`Section 5`) — calls `orchestrate.run_manifest_registration`:
   - resource plan + MINT annotation
   - TWDB landing page + PDF report map-reduce (Capability B)
   - Three-persona metadata loop (Capability D)
   - SUBSIDE field mapping (`type=subside_dataset`)
   - Link resources (TWDB landing page + GAM report PDF)
   - **Dry-run by default** (`APPLY_CHANGES = False`).  Set `APPLY_CHANGES = True` to apply.

### Path B  (Legacy — local-mount per-submodel loop)
The original per-submodel loop (`Section 6`) is preserved unchanged as a fallback.

> **Safety default:** `APPLY_CHANGES = False` in both paths.  No CKAN writes occur unless you explicitly set `APPLY_CHANGES = True`.


In [ ]:
import sys
from pathlib import Path as _Path
_REPO_ROOT = _Path(__file__).resolve().parents[1] if '__file__' in dir() else _Path.cwd().parent
if str(_REPO_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(_REPO_ROOT / 'src'))

import json
import os
import re
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

from IPython.display import Markdown, display
from dotenv import load_dotenv

from gam_registration import utils as u

load_dotenv()


def show_md(text: str) -> None:
    display(Markdown(text))


show_md("""
**Environment Ready**
- Running in standalone mode with local `utils.py`
- Manifest-driven batch registration for TWDB GAM packages
""")


## 1) Configuration

Update values in this cell or via environment variables.

Safety defaults:
- `APPLY_CHANGES = False`
- `UPLOAD_RESOURCES = False`


In [ ]:
MANIFEST_PATH = _REPO_ROOT / "data" / "twdb_gam_modflow_locations_with_bbox_strings.json"
PACKAGE_FILTERS: list[str] = []
MAX_MODELS: int | None = None
MAX_FILES_PER_MODEL = int(os.getenv("MAX_FILES_PER_MODEL", "5000"))
STOP_ON_ERROR = False

MINT_ENABLE_RESOURCE_SVO = os.getenv("MINT_ENABLE_RESOURCE_SVO", "true").strip().lower() in {"1", "true", "yes"}
MINT_API_BASE = os.getenv("MINT_API_BASE", "https://api.models.mint.tacc.utexas.edu/v2.0.0").strip()
MINT_API_TOKEN = os.getenv("MINT_API_TOKEN", "").strip() or None
MINT_RESOURCE_FIELD = os.getenv("MINT_RESOURCE_FIELD", "mint_standard_variables").strip() or "mint_standard_variables"
MINT_VALUE_MODE = os.getenv("MINT_VALUE_MODE", "name").strip() or "name"
MINT_VERIFY_SSL = os.getenv("MINT_VERIFY_SSL", "true").strip().lower() in {"1", "true", "yes"}

CKAN_URL = os.getenv("CKAN_URL", "https://ckan.tacc.utexas.edu").strip()
CKAN_OWNER_ORG = 'twdb-gams'
CKAN_OWNER_ORG_ID = os.getenv("CKAN_OWNER_ORG_ID", "8cdc42f9-2327-4fba-aa65-094579980dba").strip()

CKAN_AUTH_MODE = os.getenv("CKAN_AUTH_MODE", "tapis_password").strip()
CKAN_API_TOKEN = os.getenv("CKAN_API_TOKEN", os.getenv("CKAN_API_KEY", "")).strip()
CKAN_USERNAME = os.getenv("CKAN_USERNAME", "").strip()
CKAN_PASSWORD = os.getenv("CKAN_PASSWORD", "")
CKAN_TAPIS_URL = os.getenv("CKAN_TAPIS_URL", u.DEFAULT_TAPIS_URL).strip()

OPENAI_BASE_URL = os.getenv("OPENAI_BASE_URL", "https://ai.tejas.tacc.utexas.edu").strip() or None
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "").strip()
CKAN_LLM_MODEL = os.getenv("CKAN_LLM_MODEL", os.getenv("TOPIC_LABELER_MODEL", "Meta-Llama-3.3-70B-Instruct")).strip()

FORCE_DATASET_METADATA_UPDATE = True
APPLY_CHANGES = False
UPLOAD_RESOURCES = False
REMOVE_STALE_RESOURCES = False

CKAN_DATASET_LICENSE_ID = os.getenv("CKAN_DATASET_LICENSE_ID", "cc-by").strip() or None
CKAN_DATASET_MAINTAINER = os.getenv("CKAN_DATASET_MAINTAINER", "William Mobley").strip() or None
CKAN_DATASET_MAINTAINER_EMAIL = os.getenv("CKAN_DATASET_MAINTAINER_EMAIL", "wmobley@tacc.utexas.edu").strip() or None
CKAN_DATASET_AUTHOR = os.getenv("CKAN_DATASET_AUTHOR", "Texas Water Development Board").strip() or None
CKAN_DATASET_AUTHOR_EMAIL = os.getenv("CKAN_DATASET_AUTHOR_EMAIL", "").strip() or None
CKAN_DATASET_VERSION = os.getenv("CKAN_DATASET_VERSION", "").strip() or None
CKAN_DATASET_TYPE = os.getenv("CKAN_DATASET_TYPE", "dataset").strip() or "dataset"
CKAN_DATASET_ISOPEN = os.getenv("CKAN_DATASET_ISOPEN", "true").strip().lower() in {"1", "true", "yes"}
CKAN_TEMPORAL_COVERAGE_START = os.getenv("CKAN_TEMPORAL_COVERAGE_START", "").strip() or None
CKAN_TEMPORAL_COVERAGE_END = os.getenv("CKAN_TEMPORAL_COVERAGE_END", "").strip() or None

show_md(f"""
## Configuration Summary
- **MANIFEST_PATH:** `{MANIFEST_PATH}`
- **PACKAGE_FILTERS:** `{PACKAGE_FILTERS or '<all GAM packages>'}`
- **MAX_MODELS:** `{MAX_MODELS}`
- **MAX_FILES_PER_MODEL:** `{MAX_FILES_PER_MODEL}`
- **CKAN_URL:** `{CKAN_URL}`
- **CKAN_OWNER_ORG:** `{CKAN_OWNER_ORG}`
- **CKAN_OWNER_ORG_ID:** `{CKAN_OWNER_ORG_ID}`
- **LLM model:** `{CKAN_LLM_MODEL}`
- **OPENAI_API_KEY set:** `{bool(OPENAI_API_KEY)}`
- **APPLY_CHANGES:** `{APPLY_CHANGES}`
- **UPLOAD_RESOURCES:** `{UPLOAD_RESOURCES}`
- **REMOVE_STALE_RESOURCES:** `{REMOVE_STALE_RESOURCES}`
- **STOP_ON_ERROR:** `{STOP_ON_ERROR}`
- **MINT_ENABLE_RESOURCE_SVO:** `{MINT_ENABLE_RESOURCE_SVO}`
- **MINT_RESOURCE_FIELD:** `{MINT_RESOURCE_FIELD}`
- **MINT_VALUE_MODE:** `{MINT_VALUE_MODE}`
- **MINT_API_BASE:** `{MINT_API_BASE}`
""")

# ===========================================================================
# Capability A/B/D — new pipeline settings (added for SUBSIDE pipeline)
# ===========================================================================

# Root directory of the GAM collection on Lonestar6 / Corral filesystem.
# Example: /corral-repl/tacc/aci/PT2050/projects/PTDATAX-286/twdb_gam_collection
GAM_ROOT_DIR = '/corral-repl/tacc/aci/PT2050/projects/PTDATAX-272/Yegua-Jackson_Aquifer_GAM/'

# Generated manifest path (written by Section 3.5 discovery; can be hand-edited).
GENERATED_MANIFEST_PATH = Path("twdb_gam_manifest_generated.json")

# Maximum persona-loop rounds before flagging as unconverged.
MAX_ROUNDS = int(os.getenv("MAX_ROUNDS", "3"))

# Inter-call sleep between successive LLM calls (seconds).
# Applied between persona-loop calls and PDF map-reduce chunk calls.
LLM_CALL_DELAY_SECONDS = float(os.getenv("LLM_CALL_DELAY_SECONDS", "4"))

# SUBSIDE package type (required for subside_dataset scheming schema).
CKAN_DATASET_TYPE_SUBSIDE = "subside_dataset"

# Whether to run batched LLM resource-description review (Data Curator); default True.
# Set to False to skip LLM enrichment of individual resource descriptions.
REVIEW_RESOURCES = True

# Register-by-reference mode: when True, files stay on Corral; Tapis postit URLs are
# minted per file and registered as CKAN url-type resources (no byte uploads).
# Requires: CKAN_AUTH_MODE=tapis_password (auth_header must be a Bearer JWT),
#           TAPIS_SYSTEM_ID and TAPIS_SYSTEM_ROOTDIR env vars set.
# See docs/tapis-system-setup.md for setup instructions.
REGISTER_BY_REFERENCE = os.getenv("REGISTER_BY_REFERENCE", "false").strip().lower() in {"1", "true", "yes"}

show_md(f"""
## Configuration Summary (NEW — Capability A/B/D)
- **GAM_ROOT_DIR:** `{GAM_ROOT_DIR or '<NOT SET — needed for discovery>'}`
- **GENERATED_MANIFEST_PATH:** `{GENERATED_MANIFEST_PATH}`
- **MAX_ROUNDS (persona loop):** `{MAX_ROUNDS}`
- **LLM_CALL_DELAY_SECONDS:** `{LLM_CALL_DELAY_SECONDS}`
- **CKAN_DATASET_TYPE_SUBSIDE:** `{CKAN_DATASET_TYPE_SUBSIDE}`
- **REVIEW_RESOURCES:** `{REVIEW_RESOURCES}`
- **REGISTER_BY_REFERENCE:** `{REGISTER_BY_REFERENCE}`  _(True = postit URLs; requires TAPIS_SYSTEM_ID + Bearer JWT)_
""")

## 2) Helpers


In [ ]:
# --- New Capability A/B/D module imports ---
from gam_registration.discovery import discover_gam_models_from_local
from gam_registration.orchestrate import run_manifest_registration, run_registration, RegistrationResult
from gam_registration import subside_mapping
from gam_registration import persona_loop
from gam_registration import twdb_enrich

clean_text = u.clean_text
slugify = u.slugify
dedupe_tags = u.dedupe_tags
sanitize_tag = u.sanitize_tag
fetch_source_metadata = u.fetch_source_metadata
list_resource_files = u.list_resource_files
build_resource_plan = u.build_resource_plan
summarize_extensions = u.summarize_extensions
fetch_existing_dataset_or_none = u.fetch_existing_dataset_or_none
compare_dataset_metadata = u.compare_dataset_metadata
upsert_resources = u.upsert_resources
remove_stale_resources = u.remove_stale_resources
render_changes_table_markdown = u.render_changes_table_markdown
infer_model_label_from_resources = u.infer_model_label_from_resources
infer_standard_variable_names_from_resource_files = u.infer_standard_variable_names_from_resource_files
get_mint_standard_variables_for_model = u.get_mint_standard_variables_for_model
annotate_resource_plan_with_mint_standard_variables = u.annotate_resource_plan_with_mint_standard_variables


def load_manifest_models(path: Path) -> list[dict[str, Any]]:
    payload = json.loads(path.read_text())
    models = payload.get("models", [])
    if not isinstance(models, list):
        raise ValueError(f"Manifest at {path} does not contain a valid 'models' list.")
    return [item for item in models if isinstance(item, dict)]


def record_has_gam(record: dict[str, Any]) -> bool:
    text = " ".join(
        [
            clean_text(record.get("package_folder_name")),
            clean_text(record.get("title")),
            clean_text(record.get("twdb_page_type")),
        ]
    ).lower()
    return "gam" in text


def filter_manifest_models(models: list[dict[str, Any]], filters: list[str] | None = None) -> list[dict[str, Any]]:
    selected = []
    normalized_filters = [clean_text(value).lower() for value in (filters or []) if clean_text(value)]

    for record in models:
        if not record_has_gam(record):
            continue
        if not clean_text(record.get("twdb_page_url")):
            continue
        if not record.get("modflow_model_directories"):
            continue

        if normalized_filters:
            haystack = " ".join(
                [
                    clean_text(record.get("package_folder_name")),
                    clean_text(record.get("title")),
                    clean_text(record.get("package_id")),
                ]
            ).lower()
            if not any(token in haystack for token in normalized_filters):
                continue
        selected.append(record)

    return selected


def build_submodel_entries(record: dict[str, Any]) -> list[dict[str, Any]]:
    candidates: list[dict[str, Any]] = []
    seen: set[Path] = set()

    for item in record.get("modflow_model_directories") or []:
        raw = item.get("absolute_directory") if isinstance(item, dict) else item
        path = Path(clean_text(raw)).expanduser()
        if not str(path) or path in seen:
            continue
        seen.add(path)
        candidates.append(
            {
                "path": path,
                "relative_directory": clean_text(item.get("relative_directory")) if isinstance(item, dict) else "",
                "namefiles": item.get("namefiles") if isinstance(item, dict) else [],
            }
        )

    candidates = sorted(candidates, key=lambda item: str(item["path"]))
    pruned: list[dict[str, Any]] = []
    for candidate in candidates:
        path = candidate["path"]
        try:
            if any(path.is_relative_to(existing["path"]) for existing in pruned):
                continue
        except AttributeError:
            if any(str(path).startswith(f"{existing['path']}{os.sep}") or path == existing["path"] for existing in pruned):
                continue
        if path.exists() and path.is_dir():
            pruned.append(candidate)

    return pruned


def collect_resource_files(model_dir: Path, max_files: int) -> list[Path]:
    return list_resource_files(model_dir, max_files=max_files)


def _humanize_path_part(value: str) -> str:
    text = clean_text(value).replace("_", " ").replace("-", " ")
    text = re.sub(r"\s+", " ", text).strip()
    return text


def build_submodel_label(record: dict[str, Any], package_root: Path, submodel_entry: dict[str, Any]) -> str:
    relative_directory = clean_text(submodel_entry.get("relative_directory"))
    if relative_directory:
        rel_path = Path(relative_directory)
        parts = list(rel_path.parts)
        package_folder_name = clean_text(record.get("package_folder_name"))
        if parts and clean_text(parts[0]) == package_folder_name:
            parts = parts[1:]
    else:
        try:
            parts = list(submodel_entry["path"].relative_to(package_root).parts)
        except Exception:
            parts = list(submodel_entry["path"].parts[-3:])

    ignored = {
        "model file",
        "geodatabase",
        "modflow",
        "gwvistas",
        "input",
    }
    filtered = [_humanize_path_part(part) for part in parts if _humanize_path_part(part).lower() not in ignored]
    if not filtered:
        fallback = _humanize_path_part(submodel_entry["path"].name)
        return fallback or "default submodel"
    tail = filtered[-3:]
    return " / ".join(tail)


def build_preferred_dataset_title(record: dict[str, Any], submodel_label: str | None = None) -> str:
    title = clean_text(record.get("title") or record.get("package_folder_name"))
    if not title:
        title = "TWDB Groundwater Availability Model"
    title = title.replace(" GAM", "").replace(" gam", "")
    if submodel_label:
        return f"{title} - {submodel_label} Groundwater Availability Model Files"
    if "groundwater availability model files" in title.lower():
        return title
    return f"{title} Groundwater Availability Model Files"


def infer_model_version(record: dict[str, Any], model_dirs: list[Path]) -> str | None:
    candidates = [
        clean_text(record.get("title")),
        clean_text(record.get("package_folder_name")),
        clean_text(record.get("package_id")),
    ] + [str(path) for path in model_dirs]

    version_patterns = [
        r"version[ _-]*([0-9]+(?:\.[0-9]+)+)",
        r"\bv([0-9]+(?:\.[0-9]+)+)\b",
        r"\b([0-9]+(?:\.[0-9]+)+)\b",
    ]
    scenario_patterns = [
        r"\b(\d{4}-\d{4}(?:_[a-z0-9]+)?)\b",
        r"\b(\d{4}-(?:dor|aver|av))\b",
        r"\b(steady[ -]?state)\b",
        r"\b(transient)\b",
        r"\b(calibration)\b",
        r"\b(historical)\b",
        r"\b(predictive)\b",
    ]

    for text in candidates:
        value = clean_text(text)
        if not value:
            continue
        for pattern in version_patterns:
            match = re.search(pattern, value, flags=re.IGNORECASE)
            if match:
                return match.group(1)

    for text in candidates:
        value = clean_text(text)
        if not value:
            continue
        for pattern in scenario_patterns:
            match = re.search(pattern, value, flags=re.IGNORECASE)
            if match:
                return clean_text(match.group(1)).replace('_', '-')

    for path in model_dirs:
        name = clean_text(path.name)
        if name:
            return name.replace('_', '-')
    return None


def build_descriptive_notes(
    llm_notes: str,
    *,
    submodel_label: str,
    model_version: str | None,
    source_url: str,
    source_title: str,
    package_folder: str,
    model_dirs: list[Path],
    resource_count: int,
) -> str:
    notes_lines = [
        llm_notes,
        "",
        f"Registered submodel directory: {submodel_label}",
        f"Registered model version: {model_version or 'not explicitly identified'}",
        f"Source metadata URL: {source_url}",
        f"Source page title: {source_title}",
        f"Package folder: {package_folder}",
        f"MODFLOW directories included: {len(model_dirs)}",
        f"Resource count: {resource_count}",
        f"Metadata generated UTC: {datetime.now(timezone.utc).isoformat()}",
    ]
    return clean_text("\n".join(notes_lines), 3000)


def build_mint_candidate_model_labels(record: dict[str, Any]) -> list[str]:
    labels = [
        "MODFLOW",
        "MODFLOW-2000",
        "MODFLOW-96",
        "MODFLOW-USG",
        "MODFLOW 6",
    ]
    title = clean_text(record.get("title"))
    package_name = clean_text(record.get("package_folder_name"))
    if "usg" in f"{title} {package_name}".lower() and "MODFLOW-USG" not in labels:
        labels.insert(0, "MODFLOW-USG")
    return [label for label in dict.fromkeys(labels) if clean_text(label)]


def summarize_mint_annotation(resource_plan: list[dict[str, Any]], field_name: str) -> tuple[int, list[str]]:
    annotated = 0
    examples: list[str] = []
    for item in resource_plan:
        value = clean_text(item.get(field_name))
        if not value:
            continue
        annotated += 1
        if len(examples) < 5:
            examples.append(f"{item.get('resource_name')}: {value}")
    return annotated, examples


def summarize_batch_results(results: list[dict[str, Any]]) -> str:
    lines = [
        "| package | submodel | dataset | status | files | created | updated | removed | message |",
        "|---|---|---|---|---:|---:|---:|---:|---|",
    ]
    for item in results:
        lines.append(
            "| {package} | {submodel} | {dataset} | {status} | {files} | {created} | {updated} | {removed} | {message} |".format(
                package=str(item.get("package_folder_name", "")).replace("|", "\\|"),
                submodel=str(item.get("submodel_label", "")).replace("|", "\\|"),
                dataset=str(item.get("dataset_name", "")).replace("|", "\\|"),
                status=str(item.get("status", "")).replace("|", "\\|"),
                files=item.get("resource_count", 0),
                created=item.get("resources_created", 0),
                updated=item.get("resources_updated", 0),
                removed=item.get("resources_removed", 0),
                message=str(item.get("message", "")).replace("|", "\\|"),
            )
        )
    return "\\n".join(lines)

def _show_agent_discourse(result):
    """Render the persona-loop discourse: per round, the Domain Expert's field
    changes (its response to feedback) then each evaluator's verdict, questions,
    and recommendations."""
    _plr = getattr(result, "persona_loop_result", None)
    if _plr is None or not getattr(_plr, "transcript", None):
        return
    _lines = ["**Agent discourse**"]
    _prev = {}
    for _rnd in _plr.transcript:
        _lines.append(f"*Round {_rnd.round_number}*")
        _cand = _rnd.candidate_metadata or {}
        _fields = {k: v for k, v in _cand.items() if not k.startswith("_gap_")}
        if _rnd.round_number == 1:
            _lines.append(f"- ✍️ **Domain Expert** authored {len(_fields)} fields")
        else:
            _chg = []
            for _k, _v in _fields.items():
                if _prev.get(_k) != _v:
                    _vs = str(_v).replace("|", "\\|")
                    if len(_vs) > 80:
                        _vs = _vs[:80] + "…"
                    _chg.append(f"`{_k}` → {_vs}")
            if _chg:
                _lines.append("- ✍️ **Domain Expert** revised:")
                _lines.extend(f"    - {c}" for c in _chg)
            else:
                _lines.append("- ✍️ **Domain Expert** made no field changes this round")
        _prev = dict(_fields)
        for _who, _ev in (("Data Curator (FAIR)", _rnd.fair_evaluator),
                          ("Data Scientist (usability)", _rnd.usability_evaluator)):
            _lines.append(f"- **{_who}** — verdict: `{_ev.verdict}`")
            for _qq in (_ev.questions or []):
                _lines.append(f"    - ❓ {_qq}")
            for _rr in (_ev.recommendations or []):
                _lines.append(f"    - 💡 {_rr}")
    show_md("\n".join(_lines))


## 3) Build CKAN Auth Header


In [ ]:
auth_header = u.build_ckan_auth_header(
    auth_mode=CKAN_AUTH_MODE,
    api_token=CKAN_API_TOKEN,
    username=CKAN_USERNAME,
    password=CKAN_PASSWORD,
    tapis_url=CKAN_TAPIS_URL,
)

show_md(f"""
**CKAN Auth**
- mode: `{CKAN_AUTH_MODE}`
- auth header ready: `{bool(auth_header)}`
""")


## 3.5) [NEW] Capability A — Generate / Refresh Manifest

Recursively walks `GAM_ROOT_DIR` to find MODFLOW `.nam` files and produce a fresh manifest.

**Package detection.** If `GAM_ROOT_DIR` points at a **single GAM folder** whose children are generic component dirs (`Geodatabase/`, `Model File/`, …), the whole root is treated as **one** package, named from the folder (e.g. `Yegua-Jackson_Aquifer_GAM` → `yegua-jackson-aquifer-gam`). If it points at a **collection** of GAM folders, each child becomes its own package. This is auto-detected; override with `GAM_SINGLE_PACKAGE=auto|true|false` (default `auto`).

**Bbox derivation chain per model:**
1. **DIS via `flopy`** — extracts xoff/yoff/NROW/NCOL/DELR/DELC. Texas-bounds guard (lon −107..−93, lat 25..37) + zero-origin guard. Status: `ok_from_dis` (or `suspicious_dis` if rejected → falls through).
2. **Geodatabase (`.gdb`)** — reads the model boundary/extent from a geodatabase via geopandas (lazy import; skipped if geopandas is not installed), reprojecting to WGS84. Status: `ok_from_gdb`.
3. **Aquifer-boundary fallback (`aquifer.py`)** — looks up `gam_aquifer_map.json` by `package_id`, fetches the TWDB ArcGIS polygon, computes bbox as min/max. Status: `ok_from_aquifer`.
4. **Null spatial** — all sources failed. Status: `failed_no_spatial`.

After auto-discovery, `gam_manifest_overrides.json` overrides are merged (manual corrections survive regeneration).

> **Skip this section** if you already have a good manifest at `GENERATED_MANIFEST_PATH` or `MANIFEST_PATH`.
> **Required:** `GAM_ROOT_DIR` must be set and the path must be readable.


In [ ]:
# ---------------------------------------------------------------------------
# Section 3.5 — Discovery: generate a fresh manifest from GAM_ROOT_DIR
# ---------------------------------------------------------------------------
# Package detection mode: "auto" (default) lets discovery decide whether the
# root is a single GAM (generic component children) or a collection of GAMs.
# Override with GAM_SINGLE_PACKAGE=true|false.
_gam_single_env = os.getenv("GAM_SINGLE_PACKAGE", "auto").strip().lower()
_single_pkg = {
    "auto": None, "": None,
    "true": True, "1": True, "yes": True,
    "false": False, "0": False, "no": False,
}.get(_gam_single_env, None)

if not GAM_ROOT_DIR:
    show_md(
        "> **SKIP** — `GAM_ROOT_DIR` is not set.  "
        "Set it in `.env` or above and re-run this cell to generate a fresh manifest.  "
        "Using existing manifest instead."
    )
    _discovery_manifest = None
else:
    _gam_root = Path(GAM_ROOT_DIR)
    if not _gam_root.exists():
        show_md(f"> **WARNING** — `GAM_ROOT_DIR` path does not exist on this host: `{_gam_root}`  "
                "Ensure you are running on Lonestar6 with Corral mounted.")
        _discovery_manifest = None
    else:
        _mode_label = {None: "auto-detect", True: "single-package (forced)", False: "collection (forced)"}[_single_pkg]
        show_md(f"Running discovery on `{_gam_root}` … (package detection: **{_mode_label}**)")
        _discovery_manifest = discover_gam_models_from_local(_gam_root, single_package=_single_pkg)
        _disc_models = _discovery_manifest.get("models", []) if isinstance(_discovery_manifest, dict) else _discovery_manifest
        show_md(f"Discovery complete — **{len(_disc_models)} model(s)** found.")

        # Show per-model bbox_derivation_status for human review.
        _status_rows = [
            "| package_id | title | bbox_derivation_status | twdb_page_url |",
            "|---|---|---|---|",
        ]
        for _m in _disc_models:
            _status_rows.append(
                "| {pkg} | {title} | {status} | {url} |".format(
                    pkg=str(_m.get("package_id", "")).replace("|", "\\|"),
                    title=str(_m.get("title", ""))[:60].replace("|", "\\|"),
                    status=str(_m.get("bbox_derivation_status", "")).replace("|", "\\|"),
                    url=str(_m.get("twdb_page_url", ""))[:80].replace("|", "\\|"),
                )
            )
        show_md("\n".join(_status_rows))

        _failed_spatial = [_m for _m in _disc_models if _m.get("bbox_derivation_status") == "failed_no_spatial"]
        _suspicious = [_m for _m in _disc_models if _m.get("bbox_derivation_status") == "suspicious_dis"]
        if _failed_spatial or _suspicious:
            show_md(
                f"> **Action required:** {len(_failed_spatial)} model(s) have `failed_no_spatial` "
                f"and {len(_suspicious)} have `suspicious_dis`.  "
                "Edit the generated manifest to add corrected bboxes or `report_url` values before proceeding."
            )


## 3.6) Human Review Checkpoint — Manifest

After running Section 3.5:

1. Open `twdb_gam_manifest_generated.json` (written into `GAM_ROOT_DIR` by `discover_gam_models_from_local`).
2. For each model with `bbox_derivation_status = failed_no_spatial` or `suspicious_dis`, supply a correct bbox or note it as intentionally null.
3. Fill in `twdb_page_url` and/or `report_url` for any models that have blank values.
4. Save your edits.  You can also use `gam_manifest_overrides.json` to add stable corrections without re-running discovery.

**Only proceed past this cell when the manifest is satisfactory.**


In [ ]:
# ---------------------------------------------------------------------------
# Load the generated manifest for use in the pipeline below.
# Falls back to GENERATED_MANIFEST_PATH if discovery was skipped.
# ---------------------------------------------------------------------------
if _discovery_manifest is not None:
    # Use the in-memory result from Section 3.5 (most current).
    _pipeline_manifest_models = (
        _discovery_manifest.get("models", [])
        if isinstance(_discovery_manifest, dict)
        else _discovery_manifest
    )
    show_md(f"Using in-memory discovery manifest — **{len(_pipeline_manifest_models)} model(s)**.")
elif GENERATED_MANIFEST_PATH.exists():
    _raw = json.loads(GENERATED_MANIFEST_PATH.read_text())
    _pipeline_manifest_models = _raw.get("models", _raw) if isinstance(_raw, dict) else _raw
    show_md(
        f"Loaded generated manifest from `{GENERATED_MANIFEST_PATH}` "
        f"— **{len(_pipeline_manifest_models)} model(s)**."
    )
else:
    _pipeline_manifest_models = []
    show_md(
        "> **No generated manifest available.** "
        "Either run Section 3.5 (discovery) or place a manifest at "
        f"`{GENERATED_MANIFEST_PATH}`.  "
        "The new SUBSIDE pipeline (Section 5) will be skipped."
    )


## 4) Load Manifest and Select GAM Packages


In [ ]:
manifest_models = load_manifest_models(MANIFEST_PATH)
selected_models = filter_manifest_models(manifest_models, PACKAGE_FILTERS)
if MAX_MODELS is not None:
    selected_models = selected_models[:MAX_MODELS]

selected_submodel_count = sum(len(build_submodel_entries(record)) for record in selected_models)

show_md(f"""
## Manifest Selection
- Manifest records loaded: **{len(manifest_models)}**
- GAM packages selected: **{len(selected_models)}**
- Submodel datasets selected: **{selected_submodel_count}**
""")

preview_lines = []
for record in selected_models[:10]:
    submodels = build_submodel_entries(record)
    for submodel_entry in submodels[:2]:
        submodel_label = build_submodel_label(
            record,
            Path(clean_text(record.get("package_folder"))).expanduser(),
            submodel_entry,
        )
        preview_lines.append(f"- `{record.get('package_folder_name')}` -> `{submodel_label}`")
show_md("\n".join(["**Selected Submodel Preview**"] + preview_lines))



## 5) [NEW] SUBSIDE Pipeline — run_manifest_registration

Runs the full Capability A/B/D end-to-end pipeline for every model in the generated manifest:

1. Resource plan from local files (`list_resource_files` + `build_resource_plan`)
2. MINT annotation per resource (`annotate_resource_plan_with_mint_standard_variables`)
3. TWDB enrichment — landing page fetch + report PDF discovery
4. PDF map-reduce — `pdf_extract.run_pdf_map_reduce` (Capability B); falls back to landing-page-only if no PDF
5. Three-persona loop — Domain Expert author + Data Curator (FAIR) + Data Scientist (usability) evaluators (Capability D)
6. SUBSIDE field mapping — `subside_mapping.map_to_subside_dataset` sets `type=subside_dataset`
7. Link resources — `twdb_enrich.build_link_resources` → landing page + GAM report PDF as url-type resources

**Safety gate:** `APPLY_CHANGES = False` → dry-run only.  Set `APPLY_CHANGES = True` to write to CKAN.

> Persona-loop audit transcripts are written to `runs/<model_id>_<timestamp>.json` for evaluation.


In [ ]:
# ---------------------------------------------------------------------------
# Section 5 — SUBSIDE pipeline via orchestrate.run_manifest_registration
# ---------------------------------------------------------------------------
if not OPENAI_API_KEY:
    raise ValueError("OPENAI_API_KEY is required for the SUBSIDE pipeline.")

if not _pipeline_manifest_models:
    show_md(
        "> **SKIP** — no generated manifest models loaded.  "
        "Run Section 3.5 (discovery) first or load a manifest at `GENERATED_MANIFEST_PATH`."
    )
    _registration_results = []
else:
    _manifest_to_run = _pipeline_manifest_models
    if MAX_MODELS is not None:
        _manifest_to_run = _manifest_to_run[:MAX_MODELS]

    show_md(
        f"Running SUBSIDE pipeline for **{len(_manifest_to_run)} model(s)** "
        f"(`APPLY_CHANGES={APPLY_CHANGES}`, `MAX_ROUNDS={MAX_ROUNDS}`, "
        f"`LLM_CALL_DELAY_SECONDS={LLM_CALL_DELAY_SECONDS}`)…"
    )

    Path("runs/").mkdir(parents=True, exist_ok=True)
    _approval = "REGISTER" if APPLY_CHANGES else ""

    # Organizational/config defaults — fed to the author so license / maintainer /
    # owner_org / data-contact are populated from config instead of marked unavailable.
    _org_defaults = {
        "license_id": CKAN_DATASET_LICENSE_ID,
        "author": CKAN_DATASET_AUTHOR,
        "author_email": CKAN_DATASET_AUTHOR_EMAIL,
        "maintainer": CKAN_DATASET_MAINTAINER,
        "maintainer_email": CKAN_DATASET_MAINTAINER_EMAIL,
        "owner_org": CKAN_OWNER_ORG,
        "data_contact_email": CKAN_DATASET_MAINTAINER_EMAIL,
    }

    _registration_results = run_manifest_registration(
        _manifest_to_run,
        ckan_url=CKAN_URL,
        auth_header=auth_header,
        owner_org=CKAN_OWNER_ORG,
        llm_model=CKAN_LLM_MODEL,
        llm_api_key=OPENAI_API_KEY,
        llm_base_url=OPENAI_BASE_URL,
        approval=_approval,
        max_rounds=MAX_ROUNDS,
        runs_dir=Path("runs/"),
        org_defaults=_org_defaults,
        review_resources=REVIEW_RESOURCES,
        register_by_reference=REGISTER_BY_REFERENCE,
    )

    _ok = sum(1 for r in _registration_results if r.ok)
    _fail = sum(1 for r in _registration_results if not r.ok)
    show_md(f"Pipeline complete — **{_ok}/{len(_registration_results)} OK**, {_fail} failed.")


In [ ]:
# ---------------------------------------------------------------------------
# Section 5.1 — Display per-model dry_run_summary (+ agent discourse)
# ---------------------------------------------------------------------------
for _res in _registration_results:
    _summary = _res.dry_run_summary
    _pkg = _summary.get("package_body", {})
    _extras = _summary.get("extras", [])
    _link_plan = _summary.get("link_resource_plan", [])
    _converged = _summary.get("persona_loop_converged", "n/a")
    _rounds = _summary.get("persona_loop_rounds", "n/a")
    _stop_reason = _summary.get("persona_loop_stop_reason", "n/a")
    _questions = _summary.get("outstanding_questions", [])

    _status_icon = "OK" if _res.ok else "FAILED"
    show_md(f"### [{_status_icon}] `{_res.model_id}`")
    if not _res.ok:
        show_md(f"> **Error:** {_res.error}")
        continue

    show_md(
        f"**Package body (dataset columns)**\n"
        f"- type: `{_pkg.get('type', '')}`\n"
        f"- name: `{_pkg.get('name', '')}`\n"
        f"- title: {_pkg.get('title', '')}\n"
        f"- url: `{_pkg.get('url', '') or '<none>'}`\n"
        f"- temporal: `{_pkg.get('temporal_coverage_start','<none>')}` → `{_pkg.get('temporal_coverage_end','<none>')}`\n"
        f"- mint_standard_variables: `{_pkg.get('mint_standard_variables', [])}`"
    )

    if _extras:
        _ex_lines = ["**Extras (SUBSIDE fields)**", "| key | value |", "|---|---|"]
        for _e in _extras:
            _v = str(_e.get("value", ""))
            if len(_v) > 140:
                _v = _v[:140] + "…"
            _ex_lines.append("| {k} | {v} |".format(
                k=str(_e.get("key", "")).replace("|", "\\|"),
                v=_v.replace("|", "\\|"),
            ))
        show_md("\n".join(_ex_lines))
    else:
        show_md("_No extras._")

    if _link_plan:
        _lr = ["**Link resources (url-type)**", "| name | url | format |", "|---|---|---|"]
        for _l in _link_plan:
            _lr.append("| {n} | {u} | {f} |".format(
                n=str(_l.get("name", "")).replace("|", "\\|"),
                u=str(_l.get("url", "")).replace("|", "\\|"),
                f=str(_l.get("format", "")).replace("|", "\\|"),
            ))
        show_md("\n".join(_lr))
    else:
        show_md("_No link resources._")

    show_md(f"**Persona loop** — converged: `{_converged}` | rounds: `{_rounds}` | stop_reason: `{_stop_reason}`")
    if not _converged and _questions:
        show_md("**Outstanding questions:**\n" + "\n".join(f"- {q}" for q in _questions))

    _show_agent_discourse(_res)

    _ds_url = f"{CKAN_URL.rstrip('/')}/dataset/{_pkg.get('name', '')}"
    if _res.apply_result:
        show_md(f"**Applied ✓** — view: {_res.apply_result.get('dataset_url') or _ds_url}")
    else:
        show_md(f"_Dry-run only (`APPLY_CHANGES=False`). If applied, dataset would be at:_ {_ds_url}")


## 5.2) [NEW] Single-Model Dry Run — one set of files

Run the full SUBSIDE pipeline for **one** model and review the result **without writing to CKAN**.

- Set `SINGLE_MODEL_PACKAGE_ID` to the `package_id` (or dataset name) you want, or leave it `None` to use the first available model.
- This cell is **always a dry-run** (`approval=""`), independent of `APPLY_CHANGES` — it never writes to CKAN.
- It still reads the model's local files, fetches the TWDB landing/report pages, and calls the LLM (map-reduce + persona loop). The persona-loop transcript is written to `runs/<model_id>_<timestamp>.json`.
- To register this one model for real, use **Section 5** with `APPLY_CHANGES = True` (optionally narrow the manifest first).


In [ ]:
# ---------------------------------------------------------------------------
# Section 5.2 — Single-model run (one set of files)
# ---------------------------------------------------------------------------
# Runs the full Capability A/B/D pipeline for ONE model. Dry-run by default;
# set SINGLE_MODEL_APPLY = True to actually register THIS one model to CKAN.

SINGLE_MODEL_PACKAGE_ID = None   # package_id/name; None = first available
SINGLE_MODEL_APPLY = False       # True → write this one model to CKAN

if not OPENAI_API_KEY:
    raise ValueError("OPENAI_API_KEY is required for the single-model run.")

_candidate_models = []
try:
    if _pipeline_manifest_models:
        _candidate_models = _pipeline_manifest_models
except NameError:
    pass
if not _candidate_models:
    _candidate_models = filter_manifest_models(load_manifest_models(MANIFEST_PATH), PACKAGE_FILTERS)

if not _candidate_models:
    show_md("> **No models available.** Run Section 3.5 (discovery) or check `MANIFEST_PATH`/`PACKAGE_FILTERS`.")
    _single_result = None
else:
    if SINGLE_MODEL_PACKAGE_ID:
        _target = next((m for m in _candidate_models
                        if SINGLE_MODEL_PACKAGE_ID in (clean_text(m.get("package_id")), clean_text(m.get("name")))), None)
        if _target is None:
            _ids = ", ".join(clean_text(m.get("package_id")) for m in _candidate_models[:25])
            raise ValueError(f"package_id '{SINGLE_MODEL_PACKAGE_ID}' not found. Available (first 25): {_ids}")
    else:
        _target = _candidate_models[0]

    _target_id = clean_text(_target.get("package_id") or _target.get("name"))
    _approval = "REGISTER" if SINGLE_MODEL_APPLY else ""
    show_md(f"### Single-model {'APPLY' if SINGLE_MODEL_APPLY else 'DRY RUN'} — `{_target_id}`\n"
            f"_(MAX_ROUNDS={MAX_ROUNDS}, LLM_CALL_DELAY_SECONDS={LLM_CALL_DELAY_SECONDS})_")
    Path("runs/").mkdir(parents=True, exist_ok=True)

    _org_defaults = {
        "license_id": CKAN_DATASET_LICENSE_ID,
        "author": CKAN_DATASET_AUTHOR,
        "author_email": CKAN_DATASET_AUTHOR_EMAIL,
        "maintainer": CKAN_DATASET_MAINTAINER,
        "maintainer_email": CKAN_DATASET_MAINTAINER_EMAIL,
        "owner_org": CKAN_OWNER_ORG,
        "data_contact_email": CKAN_DATASET_MAINTAINER_EMAIL,
    }

    _single_result = run_registration(
        _target, ckan_url=CKAN_URL, auth_header=auth_header, owner_org=CKAN_OWNER_ORG,
        llm_model=CKAN_LLM_MODEL, llm_api_key=OPENAI_API_KEY, llm_base_url=OPENAI_BASE_URL,
        approval=_approval, max_rounds=MAX_ROUNDS, runs_dir=Path("runs/"), org_defaults=_org_defaults,
        review_resources=REVIEW_RESOURCES,
        register_by_reference=REGISTER_BY_REFERENCE,
    )

    if not _single_result.ok:
        show_md(f"> **Error:** {_single_result.error}")
    else:
        _s = _single_result.dry_run_summary
        _pkg = _s.get("package_body", {})
        _extras = _s.get("extras", [])
        show_md(
            f"**Package body (dataset columns)**\n"
            f"- type: `{_pkg.get('type', '')}`\n"
            f"- name: `{_pkg.get('name', '')}`\n"
            f"- title: {_pkg.get('title', '')}\n"
            f"- url: `{_pkg.get('url', '') or '<none>'}`\n"
            f"- temporal: `{_pkg.get('temporal_coverage_start','<none>')}` → `{_pkg.get('temporal_coverage_end','<none>')}`\n"
            f"- mint_standard_variables: `{_pkg.get('mint_standard_variables', [])}`"
        )
        if _extras:
            _ex = ["**Extras (SUBSIDE fields — incl. categories, collection_method, spatial)**",
                   "| key | value |", "|---|---|"]
            for _e in _extras:
                _v = str(_e.get("value", ""))
                if len(_v) > 200:
                    _v = _v[:200] + "…"
                _ex.append("| {k} | {v} |".format(
                    k=str(_e.get("key", "")).replace("|", "\\|"),
                    v=_v.replace("|", "\\|"),
                ))
            show_md("\n".join(_ex))
        _link_plan = _s.get("link_resource_plan", [])
        if _link_plan:
            _lr = ["**Link resources (url-type)**", "| name | url | format |", "|---|---|---|"]
            for _l in _link_plan:
                _lr.append("| {n} | {u} | {f} |".format(
                    n=str(_l.get("name", "")).replace("|", "\\|"),
                    u=str(_l.get("url", "")).replace("|", "\\|"),
                    f=str(_l.get("format", "")).replace("|", "\\|"),
                ))
            show_md("\n".join(_lr))
        show_md(f"**Persona loop** — converged: `{_s.get('persona_loop_converged','n/a')}` | "
                f"rounds: `{_s.get('persona_loop_rounds','n/a')}` | "
                f"stop_reason: `{_s.get('persona_loop_stop_reason','n/a')}`")
        _q = _s.get("outstanding_questions", [])
        if _q:
            show_md("**Outstanding questions:**\n" + "\n".join(f"- {x}" for x in _q))

        _show_agent_discourse(_single_result)

        _ds_url = f"{CKAN_URL.rstrip('/')}/dataset/{_pkg.get('name', '')}"
        if _single_result.apply_result:
            show_md(f"**Applied ✓** — view: {_single_result.apply_result.get('dataset_url') or _ds_url}")
        else:
            show_md(f"_Dry-run only — set `SINGLE_MODEL_APPLY = True` to register. If applied, dataset would be at:_ {_ds_url}")


## 6) [Legacy] Local-Mount Per-Submodel Registration Path

The original per-submodel registration loop is preserved here as a fallback.

> Use this path when:
> - You do NOT have the new manifest generated by Section 3.5, or
> - You need the per-submodel granularity of the original flow (multiple MODFLOW dirs per GAM package registered as separate datasets), or
> - You are iterating on a single submodel without the persona loop.
>
> **Safety default:** `APPLY_CHANGES = False` — no CKAN writes unless changed.

The new SUBSIDE pipeline in Section 5 should be preferred for all new registrations.


## 5) Register or Refresh CKAN Datasets


In [ ]:
# Legacy per-submodel path — DISABLED by default. The SUBSIDE pipeline
# (Section 5 / 5.2) is preferred. Set RUN_LEGACY_PATH = True to enable.
RUN_LEGACY_PATH = False
if not RUN_LEGACY_PATH:
    show_md("> **Legacy path skipped** (`RUN_LEGACY_PATH=False`). Use Section 5 / 5.2 — the SUBSIDE pipeline.")
    selected_models = []

if not OPENAI_API_KEY:
    raise ValueError("OPENAI_API_KEY is required for LLM metadata proposal.")

batch_results: list[dict[str, Any]] = []
selected_submodel_count = sum(len(build_submodel_entries(record)) for record in selected_models)
submodel_index = 0

for record in selected_models:
    package_folder_name = clean_text(record.get("package_folder_name"))
    source_metadata_url = clean_text(record.get("twdb_page_url"))
    package_root = Path(clean_text(record.get("package_folder"))).expanduser()

    try:
        submodel_entries = build_submodel_entries(record)
        if not submodel_entries:
            raise FileNotFoundError(f"No existing MODFLOW directories found for {package_folder_name}.")
    except Exception as exc:
        batch_results.append(
            {
                "package_folder_name": package_folder_name,
                "submodel_label": "",
                "dataset_name": slugify(clean_text(record.get("title") or package_folder_name)) or package_folder_name.lower(),
                "status": "error",
                "resource_count": 0,
                "resources_created": 0,
                "resources_updated": 0,
                "resources_removed": 0,
                "message": clean_text(str(exc), 240),
            }
        )
        if STOP_ON_ERROR:
            raise
        continue

    for submodel_entry in submodel_entries:
        submodel_index += 1
        model_dir = submodel_entry["path"]
        submodel_label = build_submodel_label(record, package_root, submodel_entry)
        preferred_title = build_preferred_dataset_title(record, submodel_label=submodel_label)
        preferred_name = slugify(preferred_title)

        show_md(f"## [{submodel_index}/{selected_submodel_count}] {preferred_title}")

        try:
            resource_files = collect_resource_files(model_dir, max_files=MAX_FILES_PER_MODEL)
            if not resource_files:
                raise FileNotFoundError(f"No resource files found under selected MODFLOW directory for {package_folder_name}: {model_dir}.")

            source_meta = fetch_source_metadata(source_metadata_url)
            resource_plan = build_resource_plan(resource_files, package_root, source_metadata_url)
            ext_counts = summarize_extensions(resource_files)
            inferred_model_version = infer_model_version(record, [model_dir])

            mint_model_label = ""
            mint_standard_variable_names: list[str] = []
            mint_standard_variable_ids: list[str] = []
            mint_annotation_count = 0
            mint_annotation_examples: list[str] = []
            mint_lookup_source = "disabled"
            mint_lookup_confidence = ""

            if MINT_ENABLE_RESOURCE_SVO:
                candidate_model_labels = build_mint_candidate_model_labels(record)
                mint_model_guess = infer_model_label_from_resources(
                    resource_files,
                    ext_counts,
                    candidate_model_labels=candidate_model_labels,
                    llm_model=CKAN_LLM_MODEL,
                    llm_api_key=OPENAI_API_KEY,
                    llm_base_url=OPENAI_BASE_URL,
                )
                mint_model_label = clean_text(mint_model_guess.get("model_label"))
                mint_lookup_confidence = clean_text(mint_model_guess.get("confidence"))

                fallback_svo_names = infer_standard_variable_names_from_resource_files(
                    resource_files,
                    model_label=mint_model_label,
                )
                mint_svo = get_mint_standard_variables_for_model(
                    mint_model_label,
                    mint_api_base=MINT_API_BASE,
                    mint_api_token=MINT_API_TOKEN,
                    fallback_model_standard_variable_names=fallback_svo_names,
                    fallback_labels=candidate_model_labels,
                    verify_ssl=MINT_VERIFY_SSL,
                )
                mint_standard_variable_names = mint_svo.get("standard_variable_names") or []
                mint_standard_variable_ids = mint_svo.get("standard_variable_ids") or []
                mint_lookup_source = clean_text(mint_svo.get("source")) or clean_text(mint_model_guess.get("method"))

                annotate_resource_plan_with_mint_standard_variables(
                    resource_plan,
                    standard_variable_ids=mint_standard_variable_ids,
                    standard_variable_names=mint_standard_variable_names,
                    model_label=mint_model_label,
                    value_mode=MINT_VALUE_MODE,
                    target_field=MINT_RESOURCE_FIELD,
                )
                mint_annotation_count, mint_annotation_examples = summarize_mint_annotation(resource_plan, MINT_RESOURCE_FIELD)

            source_context_resource = {
                "resource_name": "source-metadata-url.txt",
                "resource_title": clean_text(source_meta.get("title") or preferred_title, 140),
                "resource_description": clean_text(
                    f"{source_meta.get('meta_description') or source_meta.get('excerpt', '')} "
                    f"This registration is for submodel `{submodel_label}` in directory `{model_dir}` "
                    f"with version `{inferred_model_version or 'unknown'}`.",
                    1500,
                ),
                "resource_tags": ["source-metadata", "twdb", "groundwater", "gam", sanitize_tag(submodel_label)],
            }
            llm_input_plan = [source_context_resource] + [
                {
                    "resource_name": item["resource_name"],
                    "resource_title": item["resource_title"],
                    "resource_description": item["resource_description"],
                    "resource_tags": item["resource_tags"] + [sanitize_tag(submodel_label)],
                }
                for item in resource_plan
            ]

            llm_dataset = u.propose_ckan_dataset_metadata_with_llm(
                llm_input_plan,
                model=CKAN_LLM_MODEL,
                api_key=OPENAI_API_KEY,
                base_url=OPENAI_BASE_URL,
                source_metadata_url=source_metadata_url,
                preferred_dataset_name=preferred_name,
                preferred_dataset_title=preferred_title,
                preferred_dataset_author=CKAN_DATASET_AUTHOR,
                preferred_dataset_author_email=CKAN_DATASET_AUTHOR_EMAIL,
                preferred_dataset_maintainer=CKAN_DATASET_MAINTAINER,
                preferred_dataset_maintainer_email=CKAN_DATASET_MAINTAINER_EMAIL,
                preferred_dataset_license_id=CKAN_DATASET_LICENSE_ID,
                preferred_dataset_type=CKAN_DATASET_TYPE,
                preferred_dataset_isopen=CKAN_DATASET_ISOPEN,
                preferred_dataset_spatial=clean_text(record.get("dataset_spatial")) or None,
                preferred_temporal_coverage_start=CKAN_TEMPORAL_COVERAGE_START,
                preferred_temporal_coverage_end=CKAN_TEMPORAL_COVERAGE_END,
                preferred_dataset_tags=dedupe_tags(
                    ["twdb", "groundwater", "aquifer", "gam", "modflow", "model-files", submodel_label]
                    + [clean_text(record.get("package_folder_name"))]
                ),
                preserve_preferred_values=True,
            )

            desired_dataset_payload = {
                "name": preferred_name,
                "title": preferred_title,
                "notes": build_descriptive_notes(
                    llm_dataset["dataset_notes"],
                    submodel_label=submodel_label,
                    model_version=inferred_model_version or llm_dataset.get("dataset_version") or CKAN_DATASET_VERSION,
                    source_url=source_metadata_url,
                    source_title=clean_text(source_meta.get("title")),
                    package_folder=str(package_root),
                    model_dirs=[model_dir],
                    resource_count=len(resource_plan),
                ),
                "url": source_metadata_url,
                "owner_org": CKAN_OWNER_ORG_ID,
                "private": False,
                "author": llm_dataset.get("dataset_author") or CKAN_DATASET_AUTHOR,
                "author_email": llm_dataset.get("dataset_author_email") or CKAN_DATASET_AUTHOR_EMAIL,
                "maintainer": llm_dataset.get("dataset_maintainer") or CKAN_DATASET_MAINTAINER,
                "maintainer_email": llm_dataset.get("dataset_maintainer_email") or CKAN_DATASET_MAINTAINER_EMAIL,
                "license_id": llm_dataset.get("dataset_license_id") or CKAN_DATASET_LICENSE_ID,
                "version": inferred_model_version or llm_dataset.get("dataset_version") or CKAN_DATASET_VERSION,
                "type": llm_dataset.get("dataset_type") or CKAN_DATASET_TYPE,
                "isopen": CKAN_DATASET_ISOPEN if llm_dataset.get("dataset_isopen") is None else llm_dataset.get("dataset_isopen"),
                "spatial": clean_text(record.get("dataset_spatial")) or llm_dataset.get("dataset_spatial"),
                "temporal_coverage_start": llm_dataset.get("temporal_coverage_start") or CKAN_TEMPORAL_COVERAGE_START,
                "temporal_coverage_end": llm_dataset.get("temporal_coverage_end") or CKAN_TEMPORAL_COVERAGE_END,
                "tags": dedupe_tags(
                    ["twdb", "groundwater", "aquifer", "gam", "modflow", "model-files", submodel_label]
                    + [clean_text(record.get("title")), clean_text(record.get("package_folder_name"))]
                    + [tag.get("name", "") for tag in llm_dataset.get("dataset_tags", [])]
                    + [tag for item in resource_plan for tag in item.get("resource_tags", [])]
                ),
            }

            existing_dataset = fetch_existing_dataset_or_none(CKAN_URL, desired_dataset_payload["name"], auth_header)
            changes = compare_dataset_metadata(existing_dataset, desired_dataset_payload)

            show_md(f"""
**Inventory**
- Package: `{package_folder_name}`
- Submodel: `{submodel_label}`
- Dataset name: `{desired_dataset_payload['name']}`
- MODFLOW directory: `{model_dir}`
- Files discovered: **{len(resource_files)}**
- Unique extensions: **{len(ext_counts)}**
- Inferred model version: `{inferred_model_version or '<none>'}`
- MINT model label: `{mint_model_label or '<none>'}`
- MINT lookup source: `{mint_lookup_source}`
- MINT lookup confidence: `{mint_lookup_confidence or '<n/a>'}`
- Resources annotated with `{MINT_RESOURCE_FIELD}`: **{mint_annotation_count}**
""")
            if mint_annotation_examples:
                show_md("\n".join(["**MINT Annotation Examples**"] + [f"- `{line}`" for line in mint_annotation_examples]))
            show_md(render_changes_table_markdown(changes))

            dataset_after = existing_dataset
            if APPLY_CHANGES:
                must_update = FORCE_DATASET_METADATA_UPDATE or existing_dataset is None or any(row["status"] == "update" for row in changes)
                if must_update:
                    dataset_after = u.create_or_update_ckan_dataset(
                        CKAN_URL,
                        dataset_name=desired_dataset_payload["name"],
                        dataset_title=desired_dataset_payload["title"],
                        dataset_notes=desired_dataset_payload["notes"],
                        dataset_tags=desired_dataset_payload["tags"],
                        auth_header=auth_header,
                        owner_org=CKAN_OWNER_ORG_ID,
                        private=desired_dataset_payload["private"],
                        dataset_author=desired_dataset_payload["author"],
                        dataset_author_email=desired_dataset_payload["author_email"],
                        dataset_maintainer=desired_dataset_payload["maintainer"],
                        dataset_maintainer_email=desired_dataset_payload["maintainer_email"],
                        dataset_license_id=desired_dataset_payload["license_id"],
                        dataset_url=desired_dataset_payload["url"],
                        dataset_version=desired_dataset_payload["version"],
                        dataset_type=desired_dataset_payload["type"],
                        dataset_isopen=desired_dataset_payload["isopen"],
                        dataset_spatial=desired_dataset_payload["spatial"],
                        temporal_coverage_start=desired_dataset_payload["temporal_coverage_start"],
                        temporal_coverage_end=desired_dataset_payload["temporal_coverage_end"],
                        extra_fields={
                            "source_metadata_url": source_metadata_url,
                            "resource_root_path": str(package_root),
                            "modflow_model_directory_count": 1,
                            "modflow_model_directories": json.dumps([str(model_dir)]),
                            "modflow_submodel_label": submodel_label,
                            "twdb_package_id": clean_text(record.get("package_id")),
                        },
                    )
                    show_md(f"Dataset create/update applied: `{dataset_after.get('name')}`")
                else:
                    show_md("No dataset metadata changes needed.")
            else:
                show_md("Dry run: dataset create/update skipped (`APPLY_CHANGES=False`).")

            created_count = 0
            updated_count = 0
            removed_count = 0

            if dataset_after is not None and APPLY_CHANGES and UPLOAD_RESOURCES:
                uploaded, created_count, updated_count = upsert_resources(
                    CKAN_URL,
                    dataset_after,
                    resource_plan,
                    auth_header,
                )
                show_md(f"Uploaded resources: **{len(uploaded)}** (created={created_count}, updated={updated_count})")

                if REMOVE_STALE_RESOURCES:
                    latest_dataset = u.fetch_ckan_dataset(CKAN_URL, desired_dataset_payload["name"], auth_header=auth_header)
                    removed_count = remove_stale_resources(
                        CKAN_URL,
                        latest_dataset,
                        {item["resource_name"] for item in resource_plan},
                        auth_header,
                    )
                    show_md(f"Removed stale CKAN resources: **{removed_count}**")
            elif APPLY_CHANGES:
                show_md("Resource upload skipped because dataset creation/update did not return a dataset object.")
            else:
                show_md("Dry run: resource upload skipped (`APPLY_CHANGES` and `UPLOAD_RESOURCES` must both be `True`).")

            batch_results.append(
                {
                    "package_folder_name": package_folder_name,
                    "submodel_label": submodel_label,
                    "dataset_name": desired_dataset_payload["name"],
                    "status": "ok",
                    "resource_count": len(resource_plan),
                    "resources_created": created_count,
                    "resources_updated": updated_count,
                    "resources_removed": removed_count,
                    "message": "completed" if APPLY_CHANGES else "dry run completed",
                }
            )
        except Exception as exc:
            batch_results.append(
                {
                    "package_folder_name": package_folder_name,
                    "submodel_label": submodel_label,
                    "dataset_name": preferred_name,
                    "status": "error",
                    "resource_count": 0,
                    "resources_created": 0,
                    "resources_updated": 0,
                    "resources_removed": 0,
                    "message": clean_text(str(exc), 240),
                }
            )
            show_md(f"**Error:** `{clean_text(str(exc), 500)}`")
            if STOP_ON_ERROR:
                raise



## 6) Batch Summary


In [ ]:
show_md(f"""
## Complete
- Datasets attempted: **{len(batch_results)}**
- Successful: **{sum(1 for item in batch_results if item['status'] == 'ok')}**
- Failed: **{sum(1 for item in batch_results if item['status'] == 'error')}**
""")

show_md(summarize_batch_results(batch_results))

